In [ ]:
import os, cv2, random, shutil
import matplotlib.pyplot as plt
import numpy as np

from ultralytics import YOLO
from collections import Counter
from tqdm import tqdm

### I. Trực quan hóa dữ liệu đầu vào

In [ ]:
dataset_path = './dataset'
splits = ['train', 'valid', 'test']

classes = ['brown-planthopper', 'green-leafhopper', 'leaf-folder', 'rice-bug', 'stem-borer', 'whorl-maggot']

In [ ]:
image_dir = os.path.join(dataset_path, 'valid', 'images')
label_dir = os.path.join(dataset_path, 'valid', 'labels')

image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

plt.figure(figsize=(16, 16))
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]

for i, img_name in enumerate(random.sample(image_files, 9)):
    img_path = os.path.join(image_dir, img_name)
    img = cv2.imread(img_path)
    h_img, w_img, _ = img.shape

    label_path = os.path.join(label_dir, img_name.replace('.jpg', '.txt'))

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                cls_id, x_c, y_c, w, h = map(float, line.split())

                x1 = int((x_c - w/2) * w_img)
                y1 = int((y_c - h/2) * h_img)
                x2 = int((x_c + w/2) * w_img)
                y2 = int((y_c + h/2) * h_img)

                color = colors[int(cls_id)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                cv2.putText(img, classes[int(cls_id)], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img_rgb); plt.axis('off')

plt.tight_layout(); plt.show()

### II. Thống kê dữ liệu

In [ ]:
# from collections import Counter
# import numpy as np
#
# class_counts = {split: Counter() for split in splits}
# bbox_sizes = []  # để kiểm tra small object
#
# for split in splits:
#     lbl_dir = os.path.join(dataset_path, split, 'labels')
#     if not os.path.exists(lbl_dir):
#         continue
#     for fname in os.listdir(lbl_dir):
#         if not fname.endswith('.txt'):
#             continue
#         with open(os.path.join(lbl_dir, fname)) as f:
#             for line in f:
#                 parts = line.strip().split()
#                 if len(parts) < 5:
#                     continue
#                 cls_id = int(parts[0])
#                 w, h   = float(parts[3]), float(parts[4])
#                 class_counts[split][cls_id] += 1
#                 bbox_sizes.append(w * h)   # diện tích bbox chuẩn hóa
#
# # Visualize class distribution
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for ax, split in zip(axes, splits):
#     counts = [class_counts[split].get(i, 0) for i in range(len(classes))]
#     bars = ax.bar(classes, counts, color=['#e74c3c','#2ecc71','#3498db',
#                                           '#f39c12','#9b59b6','#1abc9c'])
#     ax.set_title(f'Class Distribution — {split}', fontsize=13, fontweight='bold')
#     ax.set_xticklabels(classes, rotation=30, ha='right')
#     ax.set_ylabel('Số lượng bbox')
#     for bar, cnt in zip(bars, counts):
#         ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
#                 str(cnt), ha='center', va='bottom', fontsize=9)
# plt.tight_layout()
# plt.show()
#
# # Kiểm tra imbalance ratio
# train_counts = [class_counts['train'].get(i, 0) for i in range(len(classes))]
# max_c, min_c = max(train_counts), min(train_counts)
# print(f"\n📊 Imbalance ratio (max/min): {max_c/max(min_c,1):.2f}x")
# for i, cls in enumerate(classes):
#     print(f"  {cls:25s}: {train_counts[i]:4d} bbox")

In [ ]:
# # ── 3.2 Kiểm tra kích thước object (small object problem) ──────────────────
# bbox_sizes = np.array(bbox_sizes)
# print(f"\n📐 Thống kê kích thước bbox (diện tích chuẩn hóa 0~1):")
# print(f"  Mean  : {bbox_sizes.mean():.4f}")
# print(f"  Median: {np.median(bbox_sizes):.4f}")
# print(f"  < 0.01 (very small): {(bbox_sizes < 0.01).sum()} bbox ({(bbox_sizes<0.01).mean()*100:.1f}%)")
# print(f"  < 0.05 (small)     : {(bbox_sizes < 0.05).sum()} bbox ({(bbox_sizes<0.05).mean()*100:.1f}%)")
#
# plt.figure(figsize=(8, 4))
# plt.hist(bbox_sizes, bins=50, color='steelblue', edgecolor='white')
# plt.axvline(0.01, color='red',    linestyle='--', label='Very small (0.01)')
# plt.axvline(0.05, color='orange', linestyle='--', label='Small (0.05)')
# plt.xlabel('Diện tích bbox (chuẩn hóa)'); plt.ylabel('Số lượng')
# plt.title('Phân phối kích thước bounding box')
# plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# # ── 3.3 Kiểm tra chất lượng ảnh (độ sáng, blur) ───────────────────────────
# brightness_list, blur_list = [], []
# img_dir = os.path.join(dataset_path, 'train', 'images')
#
# for fname in random.sample(os.listdir(img_dir), min(200, len(os.listdir(img_dir)))):
#     if not fname.endswith('.jpg'): continue
#     img = cv2.imread(os.path.join(img_dir, fname))
#     if img is None: continue
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     brightness_list.append(gray.mean())
#     blur_list.append(cv2.Laplacian(gray, cv2.CV_64F).var())
#
# print(f"\n💡 Độ sáng trung bình : {np.mean(brightness_list):.1f} / 255")
# print(f"🔍 Độ nét (Laplacian) : {np.mean(blur_list):.1f}")
# print(f"  → Ảnh blur (<100)  : {sum(b < 100 for b in blur_list)} / {len(blur_list)}")
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# ax1.hist(brightness_list, bins=30, color='gold');   ax1.set_title('Phân phối độ sáng')
# ax2.hist(blur_list,       bins=30, color='tomato'); ax2.set_title('Phân phối độ nét')
# plt.tight_layout(); plt.show()

### III. Tiền xử lý và tăng cường dữ liệu đầu vào

In [ ]:
def preprocess_dataset(src_path, dest_path, splits):
    if not os.path.exists(dest_path):
        os.makedirs(dest_path)

    for split in splits:
        src_img_dir = os.path.join(src_path, split, 'images')
        src_lbl_dir = os.path.join(src_path, split, 'labels')

        dest_img_dir = os.path.join(dest_path, split, 'images')
        dest_lbl_dir = os.path.join(dest_path, split, 'labels')

        os.makedirs(dest_img_dir, exist_ok=True)
        os.makedirs(dest_lbl_dir, exist_ok=True)

        if not os.path.exists(src_img_dir):
            continue

        image_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for fname in tqdm(image_files):
            label_name = os.path.splitext(fname)[0] + '.txt'
            label_path = os.path.join(src_lbl_dir, label_name)

            if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
                continue

            img = cv2.imread(os.path.join(src_img_dir, fname))
            if img is None:
                continue

            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)

            # Tăng cường độ tương phản
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            l = clahe.apply(l)
            img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

            # Lọc nhiễu
            img = cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)

            # Làm nét ảnh
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            if cv2.Laplacian(gray, cv2.CV_64F).var() < 100:
                kernel = np.array([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])
                img = cv2.filter2D(img, -1, kernel)

            cv2.imwrite(os.path.join(dest_img_dir, fname), img)
            shutil.copy2(label_path, os.path.join(dest_lbl_dir, label_name))

    print(f"Preprocessing successful. Optimized images are saved in {dest_path}.")

preprocess_dataset(dataset_path, './dataset_pre', splits)

In [ ]:
split = 'train'
test_img_dir = os.path.join('./dataset', split, 'images')
sample_fname = random.choice([f for f in os.listdir(test_img_dir) if f.endswith('.jpg')])

original_path = os.path.join('./dataset', split, 'images', sample_fname)
processed_path = os.path.join('./dataset_pre', split, 'images', sample_fname)

img_org = cv2.cvtColor(cv2.imread(original_path), cv2.COLOR_BGR2RGB)
img_pre = cv2.cvtColor(cv2.imread(processed_path), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_org)
plt.title(f"Original Image"); plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_pre)
plt.title("Preprocessed Image"); plt.axis('off')

plt.tight_layout(); plt.show()

### IV.

In [ ]:
model = YOLO('yolov11n.pt')

results = model.train(
    data='path/to/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='rice-pest-detection',
    name='preprocessed_version_1',
    save=True
)